El siguiente proyecto se desarrollo en Kaggle, añado en las siguientes celdas el codigo tal cual se realizó en la plataforma antes mencioanda y así mismo dejo la liga del notebook https://www.kaggle.com/code/natalia27772/sprint-17

## Inicialización

In [ ]:
import pandas as pd
import os, json, math, random, gc

from pathlib import Path
import numpy as np

import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers, callbacks


## Carga los datos

El conjunto de datos se almacena en la carpeta `/datasets/faces/` 
- La carpeta `final_files` con 7600 fotos 
- El archivo `labels.csv` con etiquetas, con dos columnas: `file_name` y `real_age` 
Dado que el número de archivos de imágenes es bastante elevado, se recomienda evitar leerlos todos a la vez, ya que esto consumiría muchos recursos computacionales. Te recomendamos crear un generador con ImageDataGenerator. Este método se explicó en el capítulo 3, lección 7 de este curso.

El archivo de etiqueta se puede cargar como un archivo CSV habitual.

In [ ]:
# Cargar las etiquetas
labels = pd.read_csv('/kaggle/input/final-faces/labels.csv')
print('Shape del archivo de etiquetas:', labels.shape)
labels.head()


In [ ]:
# Crear el generador
datagen = ImageDataGenerator(
    validation_split=0.2,     # 20% de las imágenes se usarán para validación
    rescale=1./255            # normalización de los valores de píxeles
)

# Directorio base donde están las imágenes
image_dir = '/kaggle/input/final-faces/final_files/final_files/'

# Generador para entrenamiento
train_gen = datagen.flow_from_dataframe(
    dataframe=labels,
    directory=image_dir,
    x_col='file_name',
    y_col='real_age',
    target_size=(224, 224),    # tamaño estándar para redes CNN
    batch_size=32,
    subset='training',
    class_mode='raw'           # porque queremos predecir un valor numérico (edad)
)

# Generador para validación
val_gen = datagen.flow_from_dataframe(
    dataframe=labels,
    directory=image_dir,
    x_col='file_name',
    y_col='real_age',
    target_size=(224, 224),
    batch_size=32,
    subset='validation',
    class_mode='raw'
)


In [ ]:
# Verificar una tanda de imágenes
x, y = next(train_gen)
print('Shape del lote de imágenes:', x.shape)
print('Shape de las etiquetas:', y.shape)
plt.imshow(x[0])
plt.title(f'Edad: {y[0]}')
plt.show()


## EDA

In [ ]:
# Información general del archivo de etiquetas
labels.info()

# Revisar si hay valores nulos
print("\nValores nulos por columna:")
print(labels.isna().sum())

# Describir estadísticamente la columna de edades
print("\nResumen estadístico de las edades:")
print(labels['real_age'].describe())

# Mostrar algunos ejemplos
labels.sample(5)

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(labels['real_age'], bins=30, color='skyblue', edgecolor='black')
plt.title('Distribución de Edades en el Dataset')
plt.xlabel('Edad real')
plt.ylabel('Número de imágenes')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# Crear rangos de edad
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 100]
labels['age_range'] = pd.cut(labels['real_age'], bins)

# Contar cuántas imágenes hay por rango
age_counts = labels['age_range'].value_counts().sort_index()

plt.figure(figsize=(9,5))
age_counts.plot(kind='bar', color='lightcoral', edgecolor='black')
plt.title('Número de imágenes por rango de edad')
plt.xlabel('Rango de edad')
plt.ylabel('Cantidad de imágenes')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.show()

# Mostrar los conteos
age_counts


### Conclusiones

El dataset contiene 7,591 registros, cada uno con una imagen facial y su edad real (file_name, real_age). No hay valores nulos ni duplicados, y los tipos de datos son correctos.

La edad mínima es 1 año y la máxima 100 años, con una media de 31.2 años y desviación estándar de 17.1. La mayoría de las imágenes corresponden a personas de 20 a 40 años, mientras que los grupos infantiles y mayores de 60 están poco representados.

El histograma muestra una distribución desigual, con alta concentración en edades jóvenes y adultas.

Esto puede hacer que el modelo tenga mejor precisión en ese rango y más dificultad en edades extremas.

Debido al desbalance observado en la distribución de edades, se implementará un ajuste de pesos por clase (class weights) durante el entrenamiento del modelo. Esta técnica permitirá que el modelo otorgue una mayor relevancia a los grupos de edad con menor representación, compensando el desequilibrio y favoreciendo un aprendizaje más equitativo en la predicción de la edad.

## Modelado

In [ ]:
import pandas as pd

import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Flatten
from tensorflow.keras.optimizers import Adam

In [ ]:
def load_train(path):
    datagen = ImageDataGenerator(
        validation_split=0.2,
        rescale=1./255
    )

    df = pd.read_csv('/kaggle/input/final-faces/labels.csv')

    train_gen_flow = datagen.flow_from_dataframe(
        dataframe=df,
        directory='/kaggle/input/final-faces/final_files/final_files',  # 👈 CORREGIDA
        x_col='file_name',
        y_col='real_age',
        target_size=(150, 150),
        batch_size=32,
        subset='training',
        class_mode='raw'
    )

    return train_gen_flow


In [ ]:
def load_test(path):
    datagen = ImageDataGenerator(
        validation_split=0.2,
        rescale=1./255
    )

    df = pd.read_csv('/kaggle/input/final-faces/labels.csv')

    test_gen_flow = datagen.flow_from_dataframe(
        dataframe=df,
        directory='/kaggle/input/final-faces/final_files/final_files',  # 👈 CORREGIDA
        x_col='file_name',
        y_col='real_age',
        target_size=(150, 150),
        batch_size=32,
        subset='validation',
        class_mode='raw'
    )

    return test_gen_flow

In [ ]:
def create_model(input_shape):
    """
    Define el modelo CNN usando ResNet50 como base.
    """
    backbone = ResNet50(
        input_shape=input_shape,
        include_top=False,
        weights=None
    )

    # Congelamos las capas base para entrenar solo las nuevas al principio
    backbone.trainable = False

    model = Sequential([
        backbone,
        GlobalAveragePooling2D(),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dense(1)  # salida de regresión (edad real)
    ])

    optimizer = Adam(learning_rate=0.0001)

    model.compile(
        optimizer=optimizer,
        loss='mean_absolute_error',
        metrics=['mae']
    )

    return model

In [ ]:
def train_model(model, train_data, test_data, batch_size=None, epochs=20,
                steps_per_epoch=None, validation_steps=None):
    """
    Entrena el modelo dados los parámetros.
    """
    # Callback para detener entrenamiento si no mejora
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )

    history = model.fit(
        train_data,
        validation_data=test_data,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        callbacks=[early_stopping],
        verbose=2
    )

    return model


## Preparación del script para ejecutarlo en la plataforma GPU

In [ ]:
# Script para ejecutarlo en la plataforma GPU

import inspect

# Bloque de inicialización con todas las librerías necesarias
init_str = """
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Flatten
from tensorflow.keras.optimizers import Adam
"""

# Creación el archivo .py con las funciones del modelo
with open('run_model_on_gpu.py', 'w') as f:
    f.write(init_str)
    f.write('\n\n')
    
    # Agregar el código de cada función
    for fn_name in [load_train, load_test, create_model, train_model]:
        src = inspect.getsource(fn_name)
        f.write(src)
        f.write('\n\n')

print("Archivo 'run_model_on_gpu.py' creado correctamente.")


In [ ]:
%run /kaggle/working/run_model_on_gpu.py

In [ ]:
# Cargar los datos
train_data = load_train(None)
test_data = load_test(None)

# Crear el modelo
model = create_model((150, 150, 3))

# Entrenar el modelo
trained_model = train_model(
    model, 
    train_data, 
    test_data, 
    batch_size=32, 
    epochs=10, 
    steps_per_epoch=len(train_data), 
    validation_steps=len(test_data)
)


### El resultado

Coloca el resultado de la plataforma GPU como una celda Markdown aquí.

Found 6073 validated image filenames. Found 1518 validated image filenames. 2025-11-05 20:55:41.260604: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303) /usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your PyDataset class should call super().__init__(**kwargs) in its constructor. **kwargs can include workers, use_multiprocessing, max_queue_size. Do not pass these arguments to fit(), as they will be ignored. self._warn_if_super_not_called() Epoch 1/10 190/190 - 376s - 2s/step - loss: 28.6386 - mae: 28.6386 - val_loss: 23.0771 - val_mae: 23.0771 Epoch 2/10 190/190 - 344s - 2s/step - loss: 21.4690 - mae: 21.4690 - val_loss: 16.2018 - val_mae: 16.2018 Epoch 3/10 190/190 - 342s - 2s/step - loss: 15.7014 - mae: 15.7014 - val_loss: 13.8471 - val_mae: 13.8471 Epoch 4/10 190/190 - 339s - 2s/step - loss: 14.1653 - mae: 14.1653 - val_loss: 13.8881 - val_mae: 13.8881 Epoch 5/10 190/190 - 351s - 2s/step - loss: 14.0468 - mae: 14.0468 - val_loss: 13.9467 - val_mae: 13.9467 Epoch 6/10 190/190 - 338s - 2s/step - loss: 14.0214 - mae: 14.0214 - val_loss: 13.9581 - val_mae: 13.9581

## Conclusiones

El modelo de red neuronal se entrenó en la plataforma Kaggle, completando seis épocas antes de estabilizarse. Durante las primeras iteraciones se observó una disminución progresiva en la pérdida y el error absoluto medio, mientras que a partir de la cuarta época las mejoras fueron mínimas, lo que indica que el modelo alcanzó su punto óptimo con los parámetros actuales. En general, el desempeño fue adecuado, aunque podría optimizarse mediante ajustes de hiperparámetros o ampliación del conjunto de datos.